In [4]:
# Importing all the libraries we'll need for this project
import pandas as pd
import numpy as np
import zipfile
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from difflib import get_close_matches


In [6]:
# Unzip and load the Netflix dataset

# Path to the zip file (make sure 'netflix_data.csv.zip' is in the same folder as this notebook)
zip_path = "netflix_data.csv.zip"

# Extract the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("netflix_data")

# Load CSV file
df = pd.read_csv("netflix_data/netflix_data.csv")

# Display first few rows
df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [7]:
# Let's check what columns we have and the dataset size
print("Shape of dataset:", df.shape)
print("\nColumns available:\n", df.columns)


Shape of dataset: (8807, 12)

Columns available:
 Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')


In [8]:
# We'll use only the important columns for content-based filtering
df_filtered = df[['title', 'director', 'cast', 'listed_in', 'description']].copy()

# Fill missing values with empty strings
for col in ['director', 'cast', 'listed_in', 'description']:
    df_filtered[col] = df_filtered[col].fillna('')

# Check a few examples
df_filtered.head(3)


,title,director,cast,listed_in,description
0,Dick Johnson Is Dead,Kirsten Johnson,,Documentaries,"As her father nears the end of his life, filmm..."
1,Blood & Water,,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...","International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...","Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [9]:
# Combine director, cast, genre, and description into a single text field called 'content'
df_filtered['content'] = (
    df_filtered['director'] + ' ' +
    df_filtered['cast'] + ' ' +
    df_filtered['listed_in'] + ' ' +
    df_filtered['description']
)

# Clean the text: make lowercase and remove special characters
df_filtered['content'] = df_filtered['content'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x.lower()))

# Show example
df_filtered[['title', 'content']].head(3)


,title,content
0,Dick Johnson Is Dead,kirsten johnson documentaries as her father n...
1,Blood & Water,ama qamata khosi ngema gail mabalane thabang ...
2,Ganglands,julien leclercq sami bouajila tracy gotoas sam...


In [10]:
# TF-IDF (Term Frequency - Inverse Document Frequency)
# It converts text into numeric form so we can compare movies mathematically.

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df_filtered['content'])

tfidf_matrix.shape


(8807, 5000)

In [11]:
# Instead of calculating similarity for all pairs (which uses a lot of memory),
# we use NearestNeighbors which gives us the most similar items efficiently.

from sklearn.neighbors import NearestNeighbors

nn_model = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model.fit(tfidf_matrix)


,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [12]:
def get_recommendations(title, n=10):
    """
    Returns top n movies similar to the given title.
    """
    # Check if movie exists
    matches = df_filtered[df_filtered['title'].str.lower() == title.lower()]
    if matches.empty:
        suggestions = get_close_matches(title.lower(), df_filtered['title'].str.lower().tolist(), n=5, cutoff=0.6)
        if suggestions:
            print(f"❌ Movie not found. Did you mean: {', '.join(suggestions)}?")
        else:
            print("❌ Movie not found in dataset.")
        return []
    
    # Get the index of the movie
    idx = matches.index[0]
    movie_vector = tfidf_matrix[idx]
    
    # Find nearest movies
    distances, indices = nn_model.kneighbors(movie_vector, n_neighbors=n+1)
    
    # Exclude the movie itself
    similar_indices = indices.flatten()[1:]
    
    # Return the titles of similar movies
    return df_filtered['title'].iloc[similar_indices].tolist()


In [13]:
# Example: Get top 10 similar movies to "Kota Factory"
recommendations = get_recommendations("Kota Factory", n=10)
print("🎬 Because you liked 'Kota Factory', you might also like:")
for i, rec in enumerate(recommendations, start=1):
    print(f"{i}. {rec}")


🎬 Because you liked 'Kota Factory', you might also like:
1. O-Negative, Love Can’t Be Designed
2. Chaman Bahaar
3. Engineering Girls
4. Khotey Sikkey
5. Betaal
6. The Big Day
7. Sadqay Tumhare
8. Super Nani
9. Little Things
10. Sotus The Series


In [14]:
print("✅ Movie Recommendation System successfully built!")
print("You can now use get_recommendations('Movie Title') to get similar movies.")


✅ Movie Recommendation System successfully built!
You can now use get_recommendations('Movie Title') to get similar movies.
